# Customer Churn Prediction and Retension Intelligence System

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/raw/IBM_Telco_customer_churn_IBM_dataset.csv")

df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns)

Shape: (7043, 33)
Columns: Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [7]:
df.dtypes.value_counts()

str        24
int64       6
float64     3
Name: count, dtype: int64

In [8]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing>0]

Churn Reason    5174
dtype: int64

In [9]:
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
missing_percent[missing_percent > 0]

Churn Reason    73.463013
dtype: float64

In [10]:
df['Churn Value'].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [11]:
pd.crosstab(df['Churn Value'], df['Churn Reason'].isna())

Churn Reason,False,True
Churn Value,,
0,0,5174
1,1869,0


Churn Reason is missing for all 5,174 non-churned customers and present for all 1,869 churned customers.

In [12]:
df.duplicated().sum()

np.int64(0)

In [14]:
df['CustomerID'].duplicated().sum()

np.int64(0)

No Duplicates entries

In [15]:
df.nunique().sort_values()

Count                   1
Country                 1
State                   1
Phone Service           2
Dependents              2
Partner                 2
Senior Citizen          2
Gender                  2
Churn Value             2
Paperless Billing       2
Churn Label             2
Multiple Lines          3
Internet Service        3
Online Backup           3
Online Security         3
Tech Support            3
Contract                3
Streaming Movies        3
Streaming TV            3
Device Protection       3
Payment Method          4
Churn Reason           20
Tenure Months          73
Churn Score            85
City                 1129
Monthly Charges      1585
Longitude            1651
Latitude             1652
Lat Long             1652
Zip Code             1652
CLTV                 3438
Total Charges        6531
CustomerID           7043
dtype: int64

In [16]:
df['Churn Value'].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

* Not much information with State, Country and Count
* The target column is imbalanced

In [17]:
df[["Churn Label", "Churn Value", "Churn Score", "Churn Reason"]].head(10)

,Churn Label,Churn Value,Churn Score,Churn Reason
0,Yes,1,86,Competitor made better offer
1,Yes,1,67,Moved
2,Yes,1,86,Moved
3,Yes,1,84,Moved
4,Yes,1,89,Competitor had better devices
5,Yes,1,78,Competitor offered higher download speeds
6,Yes,1,100,Competitor offered more data
7,Yes,1,92,Competitor made better offer
8,Yes,1,77,Competitor had better devices
9,Yes,1,97,Competitor had better devices


In [18]:
pd.crosstab(df["Churn Value"], df["Churn Label"])

Churn Label,No,Yes
Churn Value,,
0,5174,0
1,0,1869


In [19]:
df.groupby("Churn Value")["Churn Score"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn Value,,,,,,,,
0,5174.0,50.098183,17.702277,5.0,35.0,50.0,66.0,80.0
1,1869.0,82.510433,10.328570,65.0,74.0,82.0,91.0,100.0


* Churn Label  → exactly matches Churn Value
* Churn Reason → only available for churned customers
* Churn Score  → strongly related to Churn Value

In [20]:
df[["Count", "Country", "State"]].drop_duplicates()

,Count,Country,State
0,1,United States,California


In [21]:
df[["CustomerID", "Zip Code", "Lat Long", "Latitude", "Longitude"]].head()

,CustomerID,Zip Code,Lat Long,Latitude,Longitude
0,3668-QPYBK,90003,"33.964131, -118.272783",33.964131,-118.272783
1,9237-HQITU,90005,"34.059281, -118.30742",34.059281,-118.307420
2,9305-CDSKC,90006,"34.048013, -118.293953",34.048013,-118.293953
3,7892-POOKP,90010,"34.062125, -118.315709",34.062125,-118.315709
4,0280-XJGEX,90015,"34.039224, -118.266293",34.039224,-118.266293


* Count        → constant
* Country      → constant
* State        → constant
* CustomerID   → identifier
* Lat Long     → duplicates Latitude + Longitude information
* Zip Code     → location identifier

In [22]:
# Numerical Data inspection
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Count,7043.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
Zip Code,7043.0,93521.964646,1865.794555,90001.000000,92102.000000,93552.000000,95351.000000,96161.000000
Latitude,7043.0,36.282441,2.455723,32.555828,34.030915,36.391777,38.224869,41.962127
Longitude,7043.0,-119.798880,2.157889,-124.301372,-121.815412,-119.730885,-118.043237,-114.192901
Tenure Months,7043.0,32.371149,24.559481,0.000000,9.000000,29.000000,55.000000,72.000000
Monthly Charges,7043.0,64.761692,30.090047,18.250000,35.500000,70.350000,89.850000,118.750000
Churn Value,7043.0,0.265370,0.441561,0.000000,0.000000,0.000000,1.000000,1.000000
Churn Score,7043.0,58.699418,21.525131,5.000000,40.000000,61.000000,75.000000,100.000000
CLTV,7043.0,4400.295755,1183.057152,2003.000000,3469.000000,4527.000000,5380.500000,6500.000000


* Tenure Months   → 0 to 72
* Monthly Charges → 18.25 to 118.75
* Churn Value     → 0/1
* CLTV            → 2003 to 6500
* Count           → constant

In [23]:
df["Total Charges"].dtype

<StringDtype(storage='python', na_value=nan)>

In [24]:
(df["Total Charges"].astype(str).str.strip() == "").sum()

np.int64(11)

In [25]:
df["Total Charges"].head(10)

0     108.15
1     151.65
2      820.5
3    3046.05
4     5036.3
5     528.35
6      39.65
7      20.15
8    4749.15
9       30.2
Name: Total Charges, dtype: str

Total Charges is stored as string, and it contains 11 blank-string values. So Pandas doesn't recognize it as a numeric column.

In [26]:
df[df["Total Charges"].astype(str).str.strip() == ""][[
    "CustomerID",
    "Tenure Months",
    "Total Charges",
    "Churn Value"
]]

,CustomerID,Tenure Months,Total Charges,Churn Value
2234,4472-LVYGI,0,,0
2438,3115-CZMZD,0,,0
2568,5709-LVOEQ,0,,0
2667,4367-NUYAO,0,,0
2856,1371-DWPAZ,0,,0
4331,7644-OMVMY,0,,0
4687,3213-VVOLG,0,,0
5104,2520-SGTTA,0,,0
5719,2923-ARZLG,0,,0
6772,4075-WKNIU,0,,0


This strongly suggests these are new customers with no accumulated charges yet, rather than random missing data.

In [27]:
df[df["Total Charges"].astype(str).str.strip() == ""][[
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Value"
]]

,Tenure Months,Monthly Charges,Total Charges,Churn Value
2234,0,52.55,,0
2438,0,20.25,,0
2568,0,80.85,,0
2667,0,25.75,,0
2856,0,56.05,,0
4331,0,19.85,,0
4687,0,25.35,,0
5104,0,20.00,,0
5719,0,19.70,,0
6772,0,73.35,,0


The customer has just joined, so they have no accumulated charges yet, even though their monthly plan has a charge.

### Data Cleaning Decision — Total Charges

- Decision: Convert `Total Charges` to numeric and replace the 11 blank values with 0.
- Why: All 11 blank rows have `Tenure Months = 0`, indicating newly joined customers with no accumulated charges.
- Alternative: Drop the rows or impute with a statistical value.
- Why not: Dropping removes valid customers; median imputation would assign an unrealistic accumulated charge.

In [28]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

df["Total Charges"] = df["Total Charges"].fillna(0)

In [29]:
print(df["Total Charges"].dtype)
print("Missing values:", df["Total Charges"].isna().sum())

float64
Missing values: 0


In [30]:
cat_cols = df.select_dtypes(include="object").columns
cat_cols

C:\Users\Varunreddy\AppData\Local\Temp\ipykernel_2784\2960911289.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


Index(['CustomerID', 'Country', 'State', 'City', 'Lat Long', 'Gender',
       'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Churn Label', 'Churn Reason'],
      dtype='str')

In [31]:
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


CustomerID:
CustomerID
3668-QPYBK    1
9237-HQITU    1
9305-CDSKC    1
7892-POOKP    1
0280-XJGEX    1
             ..
2569-WGERO    1
6840-RESVB    1
2234-XADUH    1
4801-JZAZL    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64

Country:
Country
United States    7043
Name: count, dtype: int64

State:
State
California    7043
Name: count, dtype: int64

City:
City
Los Angeles       305
San Diego         150
San Jose          112
Sacramento        108
San Francisco     104
                 ... 
Milford             4
Calpine             4
Standish            4
Tulelake            4
Olympic Valley      4
Name: count, Length: 1129, dtype: int64

Lat Long:
Lat Long
33.964131, -118.272783    5
34.059281, -118.30742     5
34.048013, -118.293953    5
34.062125, -118.315709    5
34.039224, -118.266293    5
                         ..
40.181279, -120.392967    4
39.672813, -120.456699    4
40.346634, -120.386422    4
41.813521, -121.492666    4
39.191797, -120.212401    4
Name: count, 

* No phone service and No internet service are valid categories, not missing values.
* CustomerID, Lat Long are not normal categorical features.
* City has 1,129 categories, so one-hot encoding it directly may create a lot of features.
* Churn Label is duplicate target information and Churn Reason is outcome-related.

In [32]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    blank_count = df[col].astype(str).str.strip().eq("").sum()
    if blank_count > 0:
        print(col, blank_count)

C:\Users\Varunreddy\AppData\Local\Temp\ipykernel_2784\544470746.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


No blank-string categories were found

In [33]:
# Column Auditing
column_audit = pd.DataFrame({
    "dtype": df.dtypes,
    "unique_values": df.nunique(),
    "missing_values": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values("unique_values")

column_audit

,dtype,unique_values,missing_values,missing_percent
Count,int64,1,0,0.000000
Country,str,1,0,0.000000
State,str,1,0,0.000000
Phone Service,str,2,0,0.000000
Dependents,str,2,0,0.000000
Partner,str,2,0,0.000000
Senior Citizen,str,2,0,0.000000
Gender,str,2,0,0.000000
Churn Value,int64,2,0,0.000000
Paperless Billing,str,2,0,0.000000


In [38]:
df['CLTV']

0       3239
1       2701
2       5372
3       5003
4       5340
        ... 
7038    5306
7039    2140
7040    5560
7041    2793
7042    5097
Name: CLTV, Length: 7043, dtype: int64

## Column Selection Decisions

### Drop
- Count → constant column
- Country → constant column
- State → constant column
- CustomerID → unique identifier
- Churn Label → duplicate target information
- Churn Score → outcome-related information
- Churn Reason → known only after churn
- Lat Long → duplicates Latitude and Longitude
- Latitude → location information not required for this project
- Longitude → location information not required for this project
- Zip Code → location identifier; not suitable as a normal numerical feature

### Keep
- Tenure Months
- Monthly Charges
- Total Charges
- CLTV
- Partner
- Dependents
- Senior Citizen
- Gender
- Phone Service
- Multiple Lines
- Internet Service
- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies
- Contract
- Paperless Billing
- Payment Method

### Investigate
- City → high cardinality (1129 unique values)

In [ ]:
# Churn rate by city
city_churn = (
    df.groupby("City")["Churn Value"]
      .agg(["count", "mean"])
      .sort_values("mean", ascending=False)
)

city_churn.head(15)

,count,mean
City,,
Wrightwood,4,1.0
Tipton,4,1.0
Truckee,4,1.0
Twain,4,1.0
Johannesburg,4,1.0
Fort Jones,4,1.0
Smith River,4,1.0
Maricopa,4,1.0
Boulder Creek,4,1.0


In [ ]:
# Finding rare cities
city_churn.sort_values("count").head(15)

,count,mean
City,,
Smith River,4,1.0
Fort Jones,4,1.0
Johannesburg,4,1.0
Twain,4,1.0
Truckee,4,1.0
Tipton,4,1.0
Markleeville,4,0.0
Marysville,4,0.0
Cayucos,4,0.0


In [41]:
# Checking cities with enough customers
city_churn[city_churn["count"] >= 50].sort_values(
    "mean", ascending=False
).head(15)

,count,mean
City,,
San Diego,150,0.333333
San Francisco,104,0.298077
Los Angeles,305,0.295082
San Jose,112,0.258929
Fresno,64,0.250000
Long Beach,60,0.250000
Oakland,52,0.250000
Sacramento,108,0.240741


In [42]:
features = [
    'Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV',
    'Partner', 'Dependents', 'Senior Citizen', 'Gender',
    'Phone Service', 'Multiple Lines', 'Internet Service',
    'Online Security', 'Online Backup', 'Device Protection',
    'Tech Support', 'Streaming TV', 'Streaming Movies',
    'Contract', 'Paperless Billing', 'Payment Method'
]

X = df[features]
y = df["Churn Value"]

In [43]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:")
print(y.value_counts())

X shape: (7043, 20)
y shape: (7043,)
Target distribution:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


In [44]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)

Numerical features: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']
Categorical features: ['Partner', 'Dependents', 'Senior Citizen', 'Gender', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


C:\Users\Varunreddy\AppData\Local\Temp\ipykernel_2784\1890996255.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [45]:
print("Number of numerical features:", len(numeric_cols))
print("Number of categorical features:", len(categorical_cols))

Number of numerical features: 4
Number of categorical features: 16


In [48]:
X[numeric_cols].head(3)

,Tenure Months,Monthly Charges,Total Charges,CLTV
0,2,53.85,108.15,3239
1,2,70.70,151.65,2701
2,8,99.65,820.50,5372


In [49]:
X[categorical_cols].head(3)

,Partner,Dependents,Senior Citizen,Gender,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method
0,No,No,No,Male,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check
1,No,Yes,No,Female,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check
2,No,Yes,No,Female,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check


In [50]:
print("Missing values in X:")
print(X.isna().sum().sum())

print("Missing values in y:")
print(y.isna().sum())

print("Duplicate rows in X:")
print(X.duplicated().sum())

Missing values in X:
0
Missing values in y:
0
Duplicate rows in X:
0


In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)